# 2023–2026韩国海关前20日：留出月份官方页面与附件原件

按45个冻结月份独立查询官方海关列表，保存页面及所有链接的原始附件；仅来源，不读取SOXX标签或拟合。


In [ ]:
import hashlib,json,os,pathlib,tarfile,tempfile,subprocess,shlex,time,urllib.request,urllib.parse,urllib.error,datetime,re
import pandas as pd,qlib
from bs4 import BeautifulSoup
from qlib.workflow import R
from src.data.nas_market_data import nas_api_call,archive_nas_recorder,_settings,_ssh
root=pathlib.Path.cwd();card=root/'research/kcs_early20_2023_2026_held_original_capture_20261023';plan=json.loads((card/'plan.json').read_text());sha=lambda p:hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest();assert plan['frozen'] and not (card/'run_state.json').exists()
prior=plan['prior_source'];assert sha(root/prior['training_numeric_card']/'summary.json')==prior['training_numeric_summary_sha256'] and sha(root/prior['training_original_card']/'summary.json')==prior['training_original_summary_sha256']
months=[(y,m) for y in (2023,2024,2025) for m in range(1,13)]+[(2026,m) for m in range(1,10)];assert len(months)==45;snapshot=root/'data/snapshots/kcs_early20_2023_2026_held_original_capture_20261023';snapshot.mkdir(parents=True,exist_ok=True);q=plan['query_contract'];state={'status':'source_GETs_started','plan_sha256':sha(card/'plan.json'),'list_attempts':0,'page_attempts':0,'attachment_attempts':0,'model_fits':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
def fetch(url,maxbytes):
 start=datetime.datetime.now(datetime.timezone.utc).isoformat();headers={};body=b'';status=None;error=None;final=url
 try:
  r=urllib.request.urlopen(urllib.request.Request(url,headers={'User-Agent':q['user_agent']}),timeout=q['timeout_seconds_each']);status=r.status;final=r.url;headers={k:v for k,v in r.headers.items() if k.lower() not in ('set-cookie','set-cookie2','authorization','proxy-authenticate')};body=r.read(maxbytes+1);r.close()
 except urllib.error.HTTPError as exc:status=exc.code;final=exc.url;headers={k:v for k,v in exc.headers.items() if k.lower() not in ('set-cookie','set-cookie2','authorization','proxy-authenticate')};body=exc.read(maxbytes+1);error='HTTPError'
 except Exception as exc:error=type(exc).__name__+': '+str(exc)[:180]
 return {'url':url,'final_url':final,'started_utc':start,'ended_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'http_status':status,'content_type':headers.get('Content-Type',headers.get('content-type','')),'body_sha256':hashlib.sha256(body).hexdigest(),'body_bytes':len(body),'error':error,'headers':headers},body
def save(year,key,kind,receipt,body):
 folder=snapshot/str(year);folder.mkdir(parents=True,exist_ok=True);path=folder/(key+'.'+kind+'.response');path.write_bytes(body);head=folder/(key+'.'+kind+'.headers.json');head.write_text(json.dumps(receipt.pop('headers'),ensure_ascii=False,indent=2)+'\n');receipt['local_path']=str(path.relative_to(root));receipt['headers_path']=str(head.relative_to(root));receipt['headers_sha256']=sha(head);return receipt
def official_candidates(body,month,list_url):
 soup=BeautifulSoup(body,'html.parser');items=[];accepted=[]
 for a in soup.find_all('a'):
  href=a.get('href') or ''
  if 'pressReleaseView.do' not in href:continue
  title=a.get_text(' ',strip=True);compact=re.sub(r'\s+','',title);id=(urllib.parse.parse_qs(urllib.parse.urlsplit(href).query).get('newsId') or [None])[0]
  if not id:continue
  item={'newsId':id,'title_excerpt':title[:240],'url':urllib.parse.urljoin(list_url,href)};items.append(item);m=str(month);same_window=((f'{m}월1' in compact and '20일' in compact) or bool(re.search(rf'{m}\.1[-~∼〜]?20',compact)))
  if same_window and '수출입현황' in compact and '관세청' in compact:accepted.append(item)
 return items,accepted
receipts=[];rows=[]
for year,month in months:
 key=f'{year}-{month:02d}';params={'startDate':f'{key}-20','endDate':f'{key}-27','srchWord':'수출입','repCode':'B00003','pageIndex':1};list_url=q['list_url']+'?'+urllib.parse.urlencode(params);state['list_attempts']+=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');listing,data=fetch(list_url,q['max_list_bytes']);listing=save(year,key,'list',listing,data);candidates=[];accepted=[];list_status='transport_unknown'
 if listing['http_status']==200 and listing['body_bytes']<=q['max_list_bytes'] and 'html' in listing['content_type'].lower():
  candidates,accepted=official_candidates(data,month,list_url);list_status='unique' if len(accepted)==1 else ('ambiguous' if len(accepted)>1 else 'missing')
 listing['status']=list_status;listing['candidate_count']=len(candidates);listing['accepted_count']=len(accepted);listing['candidates']=candidates;listing['accepted']=accepted;page=None;attachments=[];links=[];page_status='not_attempted'
 if list_status=='unique':
  identity=accepted[0]['newsId'];page_url=q['page_url_template'].replace('{newsId}',identity);state['page_attempts']+=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');page,body=fetch(page_url,q['max_page_bytes']);page=save(year,key,'page',page,body);page['newsId']=identity;page_status='transport_unknown'
  if page['http_status']==200 and page['body_bytes']<=q['max_page_bytes'] and 'html' in page['content_type'].lower():
   soup=BeautifulSoup(body,'html.parser');titles=[x.get_text(' ',strip=True) for x in soup.find_all('h1')];compact=[re.sub(r'\s+','',x) for x in titles];page_status='identified' if any('수출입' in x and ('20일' in x or re.search(rf'{month}\.1[-~∼〜]?20',x)) for x in compact) else 'title_unknown';page['heading_candidates']=titles[:5]
   if page_status=='identified':
    seen=set()
    for a in soup.find_all('a'):
     href=a.get('href') or '';name=a.get_text(' ',strip=True);ext=re.search(r'\.(hwp|hwpx|pdf)\b',name,re.I);fileid=(urllib.parse.parse_qs(urllib.parse.urlsplit(href).query).get('fileId') or [None])[0]
     if 'download.do' not in href or ext is None or fileid is None or fileid in seen:continue
     seen.add(fileid);links.append({'fileId':fileid,'filename':name,'ext':ext.group(1).lower(),'url':urllib.parse.urljoin(page_url,href)})
   if page_status=='identified' and len(links)<=4:
    for link in links:
     state['attachment_attempts']+=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');r,blob=fetch(link['url'],q['max_attachment_bytes']);r=save(year,key+'_'+link['fileId'],'attachment',r,blob);r.update(link);magic={'hwp':bytes.fromhex('d0cf11e0a1b11ae1'),'hwpx':b'PK\x03\x04','pdf':b'%PDF'}[link['ext']];r['signature_ok']=bool(r['http_status']==200 and r['body_bytes']<=q['max_attachment_bytes'] and blob.startswith(magic));attachments.append(r);time.sleep(q['rate_pause_seconds'])
   elif len(links)>4:page_status='attachment_overflow_unknown'
  page['status']=page_status;page['attachment_links']=links;page['attachments']=attachments
 receipts.append({'month':key,'list':listing,'page':page});rows.append({'month':key,'newsId':accepted[0]['newsId'] if len(accepted)==1 else None,'list_status':list_status,'page_status':page_status,'attachment_links':len(links),'attachment_attempts':len(attachments),'signature_qualified_attachments':sum(x['signature_ok'] for x in attachments),'all_linked_signatures_qualified':bool(links and len(links)==len(attachments) and all(x['signature_ok'] for x in attachments))});print('captured',key,list_status,page_status,'links',len(links),'qualified',rows[-1]['signature_qualified_attachments'],flush=True);time.sleep(q['rate_pause_seconds'])
assert state['list_attempts']==45 and state['page_attempts']<=45 and state['attachment_attempts']<=180;state['status']='source_GETs_finished';(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');settings=_settings(root);archives=[]
for year in (2023,2024,2025,2026):
 folder=snapshot/str(year);year_rows=[x for x in receipts if x['month'].startswith(str(year))];assert len(year_rows)==(9 if year==2026 else 12);(folder/'year_manifest.json').write_text(json.dumps({'plan_sha256':sha(card/'plan.json'),'year':year,'months':year_rows},ensure_ascii=False,indent=2)+'\n');files=sorted(p for p in folder.iterdir() if p.is_file())
 with tempfile.TemporaryDirectory(prefix='kcs-held-'+str(year)+'-') as temp:
  archive=pathlib.Path(temp)/'source.tar.gz'
  with tarfile.open(archive,'w:gz') as t:
   for p in files:t.add(p,arcname=str(p.relative_to(root)),recursive=False)
  digest=sha(archive);receipt=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact'];assert receipt['sha256']==digest and receipt['size_bytes']==archive.stat().st_size;remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest;assert _ssh(settings['ssh'],'sha256sum '+shlex.quote(remote)).split()[0]==digest;archives.append({**receipt,'year':year,'nas_path':remote,'files':len(files),'year_manifest_sha256':sha(folder/'year_manifest.json')})
  with tempfile.TemporaryDirectory(prefix='kcs-held-cold-'+str(year)+'-') as tmp:
   copy=pathlib.Path(tmp)/'source.tar.gz'
   with copy.open('wb') as out:subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,check=True)
   assert copy.stat().st_size==receipt['size_bytes'] and sha(copy)==digest
   with tarfile.open(copy,'r:gz') as t:
    names={m.name for m in t if m.isfile()};assert names=={str(p.relative_to(root)) for p in files}
    for p in files:assert t.extractfile(str(p.relative_to(root))).read()==p.read_bytes()
 print('NAS held year cold restored',year,len(files),flush=True)
table=pd.DataFrame(rows);result={'status':'official_2023_2026_KCS_early20_held_original_page_attachment_capture_source_only','plan_sha256':sha(card/'plan.json'),'months':45,'list_GET_attempts':state['list_attempts'],'page_GET_attempts':state['page_attempts'],'attachment_GET_attempts':state['attachment_attempts'],'list_status_counts':table.list_status.value_counts().to_dict(),'page_status_counts':table.page_status.value_counts().to_dict(),'months_all_linked_signatures_qualified':int(table.all_linked_signatures_qualified.sum()),'unknown_months':table.loc[~table.all_linked_signatures_qualified,'month'].tolist(),'year_archives':archives,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n');(card/'source_restore.json').write_text(json.dumps({'status':'NAS_four_held_year_source_groups_all_members_cold_restored_exact','years':[{'year':x['year'],'sha256':x['sha256'],'files':x['files']} for x in archives]},indent=2)+'\n')
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/'data/qlib/minute_information_pit_allocator_20261012/daily_prices'),region='us',kernels=1)
with R.start(experiment_name=plan['experiment'],uri=str(card/'qlib_recorder')):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=plan,year_archives=archives);rec.save_objects(artifact_path='analysis',monthly_capture=table,source_receipts=receipts,result=result);rec.log_metrics(months=45,list_GET_attempts=state['list_attempts'],page_GET_attempts=state['page_attempts'],attachment_GET_attempts=state['attachment_attempts'],months_all_linked_signatures_qualified=result['months_all_linked_signatures_qualified'],model_fits=0,native_accounts=0);rid,eid=rec.id,str(rec.experiment_id)
state['status']='finished';state['recorder_id']=rid;state['experiment_id']=eid;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');evidence=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rid,experiment_id=eid);assert _ssh(settings['ssh'],'sha256sum '+shlex.quote(evidence['nas_path'])).split()[0]==evidence['sha256']
summary={'experiment':plan['experiment'],'decision':'KCS_2023_2026_held_original_attachment_source_capture_numeric_PIT_unqualified','objective_achieved':False,'plan_sha256':result['plan_sha256'],'result':'result.json','year_archives':archives,'source_restore':'source_restore.json','recorder_id':rid,'experiment_id':eid,'recorder_uri':str(card/'qlib_recorder'),'NAS_evidence':evidence,'NAS_readback':{'sha256':evidence['sha256'],'verified':True},'months_all_linked_signatures_qualified':result['months_all_linked_signatures_qualified'],'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/monthly_capture','analysis/source_receipts','analysis/result']},'limitations':['Current attachments and portal dates do not prove original historic first-public versions.','No semiconductor values, Qlib factor, SOXX market outcomes or funded account in this capture.','Unknown list/page/attachment months are preserved for a separately frozen repair.']};(card/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n');print(json.dumps({'decision':summary['decision'],'list_status':result['list_status_counts'],'page_status':result['page_status_counts'],'qualified_months':result['months_all_linked_signatures_qualified'],'unknown':result['unknown_months'],'attachment_GETs':state['attachment_attempts'],'recorder_id':rid,'NAS_sha256':evidence['sha256']},ensure_ascii=False,indent=2))


## 中文结论

按冻结查询合同，2023–2025全部36个月和2026年1–9月共 **45** 个月各执行一次官方海关目录GET；其中 **44个月**出现唯一原稿页且页面与全部附件签名合格。2025-08目录请求发生 **TLS握手超时、零正文**，本版本按约定保持未知，不能写作“海关未公布”。其余44页列出 **58份**原件：HWP 30、HWPX 14、PDF 14。HWPX/PDF双格式从 **2025-07** 开始出现，所有版本保留，不根据后续数值选取。

四个年度页面、附件、响应头与UTC接收时刻分别按原字节归档 Yanxuan NAS，并已**逐文件冷恢复**；原生 Qlib Recorder `328486d078f04b89b19be724feda11c0` 的45行状态和逐次收据也已NAS限定读回，SHA-256 `dbb723b702de188d2e7e90646bd11996312c26106a9f5d85e4ae36480fe0db91`。[来源质量收据](source_quality.json)记录唯一缺口的实际TLS错误和格式切换。0市场结果读取、0拟合、0资金账户。

决定 `KCS_2023_2026_held_original_attachment_source_capture_numeric_PIT_unqualified`。**44/45原件可取回不等于留出因子已合格**：2025-08需另冻一次传输修复，2025-07之后的HWPX数值需与PDF独立核对；今天可下载的文件和门户登记日期不能认证各历史月首次公开版本。后续仍须用固定原报“出口半导体同比”规则审查45个月及工作日差异，映射到Qlib决策时钟，之后才可与原Stage1输入做匹配信息检验和完整0bp账户。SOXX稳定超额尚无新证明。
